In [12]:
################################ Read and match spectroscopy classification with S1 and S2 ####
#### Author: Boya ("Paul") Zhang

import numpy as np
from matplotlib import pyplot as plt

import shapefile as shp
from shapely.geometry import Point
from shapely.geometry import shape 
import pandas as pd
import matplotlib
import os
from datetime import datetime, timedelta
#import mpld3
import plotly.graph_objects as go
from sklearn.metrics import r2_score
from scipy.stats import linregress
import matplotlib.font_manager as fm
from collections import Counter
import seaborn as sns
from osgeo import gdal
import subprocess
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import geopandas as gpd
from rasterio.mask import mask
from rasterio.plot import show
from osgeo import gdal, gdalconst, ogr
import glob
from scipy.ndimage import zoom
from rasterio.windows import Window
from rasterio.transform import Affine
from rasterio.merge import merge


In [2]:
def find_index(x, y):
    index = np.where((x_coords_flat == x) & (y_coords_flat == y))[0]
    if len(index) > 0:
        return index[0]
    else:
        return np.nan

In [3]:
### mosaic the three warped images ############
path = 'D:/Paul/Groundwater/Spectroscopy/2019/'
image_files = []
for file in os.listdir(path):
    if file.endswith('.tif'):
        image_files.append(os.path.join(path, file))


In [4]:
vrt_options = gdal.BuildVRTOptions(resampleAlg='nearest')  # Choose resampling algorithm as needed
output_vrt = path + "mosaic.vrt"
gdal.BuildVRT(output_vrt, image_files, options=vrt_options)
output_tif = path + "mosaic.tif"
gdal.Translate(output_tif, output_vrt)

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x000001F801977810> >

In [13]:
### project from wgs84 to utm10 for SAR scenes and save as tif 
ct = 0
shapefile_path = r'//pinto/Winshare/Personal/Paul/Flooding/YoloShape/Delta.shp'
shapefile = gpd.read_file(shapefile_path)


print('folder is')
os.chdir(r'D:\Paul\Groundwater\Spectroscopy\Data\S1\2019')
current_directory = os.getcwd()

# Find all .tif and .tiff files in the current directory
tif_files = glob.glob(os.path.join(current_directory, '*.tif'))
down = [file for file in tif_files if "up" not in file]
up = [file for file in tif_files if "up" in file]

file = up
for bd in range(len(file)):
    print("current band is:")
    print(bd)
    target_file = None
    
    target_file = file[bd]
    if ct == 0: ## for the first folder and the first band
        with rasterio.open(target_file) as src:
            
            # Get the profile (metadata) of the input raster
            src_profile = src.profile

            # Get the transform and dimensions of the input raster
            src_transform = src.transform
            src_width = src.width
            src_height = src.height

            # Calculate the new transform to reproject to UTM 10N
            dst_crs = 'EPSG:32610'  # UTM 10N
            dst_transform, dst_width, dst_height = calculate_default_transform(src.crs, dst_crs, src_width, src_height, *src.bounds)

            # Update the profile with the new CRS and transform
            dst_profile = src_profile.copy()
            dst_profile.update({
                'crs': dst_crs,
                'transform': dst_transform,
                'width': dst_width,
                'height': dst_height
            })
            
            file_name = os.path.splitext(os.path.basename(target_file))[0]
            with rasterio.open(file_name + '_rep.tif', 'w', **dst_profile) as dst:
                dst.write(src.read())
            
            

folder is
current band is:
0
current band is:
1


In [14]:
###### mosaic SAR by polarization ####
file_paths = ['20190924_VH_rep.tif', '20190924_up_VH_rep.tif']

# Open each raster file
src_files_to_mosaic = [rasterio.open(fp) for fp in file_paths]

# Merge the raster files
mosaic, out_trans = merge(src_files_to_mosaic)

# Get metadata of the first raster file
meta = src_files_to_mosaic[0].meta

# Update metadata with mosaic properties
meta.update({
    'height': mosaic.shape[1],
    'width': mosaic.shape[2],
    'transform': out_trans
})

# Write the mosaic to a new raster file
output_fp = 'mosaic_VH.tif'
with rasterio.open(output_fp, 'w', **meta) as dst:
    dst.write(mosaic)

RasterioIOError: 20190924_VH_rep.tif: No such file or directory

In [2]:
ct = 0
shapefile_path = r'//pinto/Winshare/Personal/Paul/Flooding/YoloShape/Delta.shp'
shapefile = gpd.read_file(shapefile_path)

print('folder is')
os.chdir(r'D:\Paul\Groundwater\Spectroscopy\Data\S1\2019')
current_directory = os.getcwd()

tif_files = glob.glob(os.path.join(current_directory, '*.tif'))

folder is


In [4]:
VV = [file for file in tif_files if "VV" in file]

In [5]:
## get mosaic sar scence cover entire Delta
## mask with Delta shapefile
## warp with s2

ct = 0
shapefile_path = r'//pinto/Winshare/Personal/Paul/Flooding/YoloShape/Delta.shp'
shapefile = gpd.read_file(shapefile_path)

print('folder is')
os.chdir(r'D:\Paul\Groundwater\Spectroscopy\Data\S1\2019')
current_directory = os.getcwd()

tif_files = glob.glob(os.path.join(current_directory, '*.tif'))
reps_up = [file for file in tif_files if "rep" in file and "up" in file]

# Filter files that contain "rep" but not "up" in their filenames

file = 'VV.tif'

with rasterio.open('D:/Paul/Groundwater/Spectroscopy/2019/mosaic_spc.tif') as src:
    # Read the image data
    s2 = src.read()
    meta_s2 = src.meta


for bd in range(1):
    print("current band is:")
    print(bd)
    target_file = None
    
    target_file = file
    if ct == 0: ## for the first folder and the first band
        with rasterio.open(target_file) as src:
            # Read the image data
            out_image, out_transform = mask(src, shapefile.geometry, crop=True)
            # Update the metadata for the cropped image
            out_meta = src.meta.copy()
            out_meta.update({'driver': 'GTiff',
                             'height': out_image.shape[1],
                             'width': out_image.shape[2],
                             'transform': out_transform})

            # Get image metadata
            meta1 = out_meta
            
            out_warp = np.empty((s2.shape[0], meta_s2['height'], meta_s2['width']), dtype=s2.dtype)
            reproject(
                source=out_image,
                destination=out_warp,
                src_transform=meta1['transform'],
                src_crs=meta1['crs'],
                dst_transform=meta_s2['transform'],
                dst_crs=meta_s2['crs'],
                resampling=Resampling.nearest
            )

            meta_warp = {
            'driver': 'GTiff',  # or any other raster format you are using
            'dtype': out_warp.dtype,
            'nodata': None,  # Adjust if there is a nodata value
            'width': out_warp.shape[2],  # Width of the reprojected raster
            'height': out_warp.shape[1],  # Height of the reprojected raster
            'count': 1,  # Number of bands
            'crs': meta_s2['crs'],  # CRS of the reprojected raster
            'transform': meta_s2['transform']  # Transformation matrix of the reprojected raster
            # Add any other metadata as needed
            }
            
        
        print(out_warp.shape)
        x, y = out_warp.shape[-2:]
        img_data_3d = np.empty((2, x, y), dtype=out_warp.dtype) ### for each image, we extract 5 bands. 
        img_data_3d[ct, :, :] = out_warp[0, :, :]  # Assuming it's a single-band grayscale image

    else:
        with rasterio.open(target_file) as src:
            # Read the image data
             # Read the image data
            out_image, out_transform = mask(src, shapefile.geometry, crop=True)
            # Update the metadata for the cropped image
            
            out_warp = np.empty((s2.shape[0], meta_s2['height'], meta_s2['width']), dtype=s2.dtype)
            reproject(
                source=out_image,
                destination=out_warp,
                src_transform=meta1['transform'],
                src_crs=meta1['crs'],
                dst_transform=meta_s2['transform'],
                dst_crs=meta_s2['crs'],
                resampling=Resampling.nearest
            )

            meta_warp = {
            'driver': 'GTiff',  # or any other raster format you are using
            'dtype': out_warp.dtype,
            'nodata': None,  # Adjust if there is a nodata value
            'width': out_warp.shape[2],  # Width of the reprojected raster
            'height': out_warp.shape[1],  # Height of the reprojected raster
            'count': 1,  # Number of bands
            'crs': meta_s2['crs'],  # CRS of the reprojected raster
            'transform': meta_s2['transform']  # Transformation matrix of the reprojected raster
            # Add any other metadata as needed
            }
            
            img_data_3d[ct, :, :] = out_warp
            

            # Get image metadata
#                 if i == 1:
#                     meta2 = src.meta
#                 else:
#                     meta3 = src.meta
    ct = ct + 1

folder is
current band is:
0
(1, 4701, 3022)


In [6]:
meta_s2

{'driver': 'GTiff',
 'dtype': 'uint8',
 'nodata': None,
 'width': 3022,
 'height': 4701,
 'count': 1,
 'crs': CRS.from_epsg(32610),
 'transform': Affine(20.0, 0.0, 595680.0,
        0.0, -20.0, 4272340.0)}

In [7]:
meta_warp

{'driver': 'GTiff',
 'dtype': dtype('uint8'),
 'nodata': None,
 'width': 3022,
 'height': 4701,
 'count': 1,
 'crs': CRS.from_epsg(32610),
 'transform': Affine(20.0, 0.0, 595680.0,
        0.0, -20.0, 4272340.0)}

In [ ]:
up = img_data_3d

In [12]:
down = img_data_3d

In [8]:
os.chdir('D:\Paul\Groundwater\Spectroscopy')
with rasterio.open('Delta_201910_mosaic_v5.tif') as src:
        # Read the image data
        spc19 = src.read()
        meta19 = src.meta

################ rasterio warp spc19 to the subset of the first folder image ######
spc_warp = np.empty((spc19.shape[0], meta_warp['height'], meta_warp['width']), dtype=spc19.dtype)
reproject(
    source=spc19,
    destination=spc_warp,
    src_transform=meta19['transform'],
    src_crs=meta19['crs'],
    dst_transform=meta_warp['transform'],
    dst_crs=meta_warp['crs'],
    resampling=Resampling.nearest
)


meta_spcw = {
            'driver': 'GTiff',  # or any other raster format you are using
            'dtype': out_warp.dtype,
            'nodata': None,  # Adjust if there is a nodata value
            'width': meta_warp['width'],  # Width of the reprojected raster
            'height': meta_warp['height'],  # Height of the reprojected raster
            'count': 1,  # Number of bands
            'crs': meta_warp['crs'],  # CRS of the reprojected raster
            'transform': meta_warp['transform']  # Transformation matrix of the reprojected raster
            # Add any other metadata as needed
            }
#meta_spcw = meta_warp.copy()
# meta_spcw.update({
#     'width': meta_warp['width'],
#     'height': meta_warp['height']
# })

In [22]:
os.chdir(r'D:\Paul\Groundwater\Spectroscopy\Data\S1\2019')
with rasterio.open('spc_warp_vv.tif', 'w', **meta_spcw) as dst:
    # Get the number of bands in the raster data
    num_bands = spc_warp.shape[0]  # Assuming the first dimension represents bands

    # Write each band of the raster data separately
    for band_idx in range(num_bands):
        dst.write(spc_warp[band_idx, :, :], band_idx + 1)  # Add 1 to band_idx to match rasterio's 1-based indexing

In [23]:
# Get the affine transformation matrix from the metadata
transform = meta_spcw['transform']

# Get the height and width of the raster data
height, width = spc_warp.shape[1:]

# Initialize arrays to store pixel coordinates
x_coords = np.empty((height, width))
y_coords = np.empty((height, width))

# Calculate pixel coordinates for all pixels in the raster
for i in range(height):
    for j in range(width):
        # Calculate pixel center coordinates
        x_coords[i, j], y_coords[i, j] = rasterio.transform.xy(transform, i, j)

# Flatten the arrays to get 1D arrays of coordinates
x_coords_flat = x_coords.flatten()
y_coords_flat = y_coords.flatten()

In [ ]:
df['index'] = df.apply(lambda row: find_index(row[' x'], row[' y']), axis=1)

In [25]:
vv_flat = img_data_3d[0, :, :].flatten()

In [17]:
df = pd.read_csv('D:\Paul\Groundwater\Spectroscopy\2019\2019.csv')

In [11]:
df.shape

NameError: name 'df' is not defined

In [ ]:

vh_flat = down[1, :, :].flatten()
spc_warp_flat = spc_warp.flatten()

mask = (spc_warp_flat == 14) | (spc_warp_flat == 10)

# Create a new array with values replaced according to the mask
flood = np.where(mask, 1, 0)

os.chdir(r'D:\Paul\Groundwater\Spectroscopy\Data\S1\2019')
ind = np.arange(0, spc_warp_flat.shape[0], 1)
out = np.vstack((ind, x_coords_flat, y_coords_flat, vv_flat, vh_flat, spc_warp_flat, flood))
out = out.transpose()

with open('warp_vv.csv',"w") as f:
    np.savetxt(f, out, delimiter=",", header="ID, x, y, VV, VH, classification, flood", 
               fmt="%f")